# Silver Layer — Snowpark Data Processing

Demonstrates Snowpark DataFrame API for complex transformations:
- Window functions for customer analytics
- Python UDFs for sentiment scoring
- Writing processed results to analytics tables

**Prerequisites:** Run phases 01–02 (setup + synthetic data generation).

In [ ]:
from snowflake.snowpark import functions as F
from snowflake.snowpark import types as T
from snowflake.snowpark.window import Window
from snowflake.snowpark.context import get_active_session

session = get_active_session()

In [ ]:
%%sql -r session_context
SELECT CURRENT_WAREHOUSE() AS WAREHOUSE,
       CURRENT_DATABASE()  AS DATABASE,
       CURRENT_SCHEMA()    AS SCHEMA

## 2. Read Bronze Source Data

In [ ]:
orders_df      = session.table("BRONZE.ORDERS")
customers_df   = session.table("BRONZE.CUSTOMERS")
products_df    = session.table("BRONZE.PRODUCTS")
reviews_df     = session.table("BRONZE.PRODUCT_REVIEWS")
order_items_df = session.table("BRONZE.ORDER_ITEMS")

print(f"Orders:    {orders_df.count():,} rows")
print(f"Customers: {customers_df.count():,} rows")
print(f"Reviews:   {reviews_df.count():,} rows")

## 3. Window Functions — Customer Purchase Patterns

Calculates running totals, order sequence, and inter-order gaps per customer.

In [ ]:
customer_window = Window.partition_by("CUSTOMER_ID").order_by("ORDER_DATE")
customer_all    = Window.partition_by("CUSTOMER_ID")

customer_orders = (
    orders_df
    .select("ORDER_ID", "CUSTOMER_ID", "ORDER_DATE", "TOTAL_AMOUNT", "REGION", "CHANNEL")
    .with_column("ORDER_SEQUENCE",
        F.row_number().over(customer_window))
    .with_column("RUNNING_TOTAL",
        F.sum("TOTAL_AMOUNT").over(customer_window.rows_between(
            Window.UNBOUNDED_PRECEDING, Window.CURRENT_ROW)))
    .with_column("AVG_ORDER_VALUE",
        F.avg("TOTAL_AMOUNT").over(customer_all))
    .with_column("DAYS_SINCE_PREV_ORDER",
        F.datediff("day",
            F.lag("ORDER_DATE").over(customer_window),
            F.col("ORDER_DATE")))
    .with_column("TOTAL_CUSTOMER_ORDERS",
        F.count("ORDER_ID").over(customer_all))
)

customer_orders.filter(F.col("CUSTOMER_ID") == 1).sort("ORDER_SEQUENCE").show(10)

## 4. Python UDF — Sentiment Scoring

Registers a permanent UDF for keyword-based sentiment analysis on product reviews.

In [ ]:
@F.udf(
    name="BRONZE.UDF_SENTIMENT_SCORE",
    is_permanent=True,
    stage_location="@ML.ML_MODELS",
    replace=True,
    packages=["snowflake-snowpark-python"],
    input_types=[T.StringType()],
    return_type=T.VariantType()
)
def sentiment_score(review_text: str) -> dict:
    """Simple keyword-based sentiment scoring for product reviews."""
    if not review_text:
        return {"score": 0.0, "label": "neutral", "confidence": 0.0}

    text = review_text.lower()

    positive_words = [
        "excellent", "amazing", "love", "great", "best", "outstanding",
        "fantastic", "perfect", "recommend", "reliable", "impressed",
        "seamless", "intuitive", "efficient", "superb", "wonderful"
    ]
    negative_words = [
        "terrible", "worst", "hate", "awful", "poor", "disappointing",
        "broken", "useless", "waste", "crash", "bug", "slow",
        "unacceptable", "frustrating", "defective", "unreliable"
    ]

    pos_count = sum(1 for w in positive_words if w in text)
    neg_count = sum(1 for w in negative_words if w in text)
    total = pos_count + neg_count

    if total == 0:
        return {"score": 0.0, "label": "neutral", "confidence": 0.3}

    score = round((pos_count - neg_count) / total, 2)
    confidence = round(min(total / 5.0, 1.0), 2)

    if score > 0.2:
        label = "positive"
    elif score < -0.2:
        label = "negative"
    else:
        label = "neutral"

    return {"score": score, "label": label, "confidence": confidence}

print("Sentiment UDF registered successfully.")

## 5. Apply Sentiment Analysis to Reviews

In [ ]:
reviews_scored = (
    reviews_df
    .with_column("SENTIMENT_RESULT",
        F.call_udf("BRONZE.UDF_SENTIMENT_SCORE", F.col("REVIEW_TEXT")))
    .with_column("SENTIMENT_SCORE",
        F.col("SENTIMENT_RESULT")["score"].cast(T.FloatType()))
    .with_column("SENTIMENT_LABEL",
        F.col("SENTIMENT_RESULT")["label"].cast(T.StringType()))
    .with_column("SENTIMENT_CONFIDENCE",
        F.col("SENTIMENT_RESULT")["confidence"].cast(T.FloatType()))
    .select(
        "REVIEW_ID", "PRODUCT_ID", "CUSTOMER_ID", "REVIEW_TEXT",
        "RATING", "REVIEW_DATE", "HELPFUL_VOTES",
        "SENTIMENT_SCORE", "SENTIMENT_LABEL", "SENTIMENT_CONFIDENCE"
    )
)

reviews_scored.write.save_as_table("GOLD.PROCESSED_REVIEWS", mode="overwrite")
print(f"Processed {reviews_scored.count():,} reviews with sentiment analysis.")

In [ ]:
%%sql -r sentiment_dist
SELECT
    SENTIMENT_LABEL,
    COUNT(*)                       AS COUNT,
    ROUND(AVG(RATING), 2)          AS AVG_RATING,
    ROUND(AVG(SENTIMENT_SCORE), 2) AS AVG_SENTIMENT_SCORE
FROM GOLD.PROCESSED_REVIEWS
GROUP BY SENTIMENT_LABEL
ORDER BY COUNT DESC

## 6. Product Sentiment Summary

In [ ]:
product_sentiment = (
    session.table("GOLD.PROCESSED_REVIEWS")
    .join(
        products_df.select("PRODUCT_ID", "PRODUCT_NAME", "CATEGORY", "BRAND"),
        on="PRODUCT_ID", how="left"
    )
    .group_by("PRODUCT_ID", "PRODUCT_NAME", "CATEGORY", "BRAND")
    .agg(
        F.count("*").alias("REVIEW_COUNT"),
        F.avg("RATING").alias("AVG_RATING"),
        F.avg("SENTIMENT_SCORE").alias("AVG_SENTIMENT"),
        F.sum(F.when(F.col("SENTIMENT_LABEL") == "positive", 1).otherwise(0))
            .alias("POSITIVE_REVIEWS"),
        F.sum(F.when(F.col("SENTIMENT_LABEL") == "negative", 1).otherwise(0))
            .alias("NEGATIVE_REVIEWS"),
        F.sum("HELPFUL_VOTES").alias("TOTAL_HELPFUL_VOTES")
    )
)

product_sentiment.write.save_as_table("GOLD.PRODUCT_SENTIMENT_SUMMARY", mode="overwrite")
print("Product sentiment summary created.")
product_sentiment.sort(F.col("REVIEW_COUNT").desc()).show(10)

## 7. Revenue Analytics with Month-over-Month Growth

In [ ]:
month_window = Window.partition_by("CATEGORY").order_by("ORDER_MONTH")

monthly_revenue = (
    orders_df
    .join(order_items_df, on="ORDER_ID", how="inner")
    .join(
        products_df.select("PRODUCT_ID", "CATEGORY", "BRAND"),
        on="PRODUCT_ID", how="left"
    )
    .with_column("ORDER_MONTH",
        F.date_trunc("month", F.col("ORDER_DATE")))
    .group_by("ORDER_MONTH", "CATEGORY")
    .agg(
        F.sum("LINE_TOTAL").alias("MONTHLY_REVENUE"),
        F.count(F.col("ORDER_ITEMS.ORDER_ID")).alias("ITEM_COUNT"),
        F.count_distinct("ORDERS.ORDER_ID").alias("ORDER_COUNT")
    )
)

revenue_with_growth = (
    monthly_revenue
    .with_column("PREV_MONTH_REVENUE",
        F.lag("MONTHLY_REVENUE").over(month_window))
    .with_column("MOM_GROWTH_PCT",
        F.when(F.col("PREV_MONTH_REVENUE").is_not_null(),
            F.round(
                (F.col("MONTHLY_REVENUE") - F.col("PREV_MONTH_REVENUE"))
                / F.col("PREV_MONTH_REVENUE") * 100, 2
            ))
        .otherwise(None))
)

revenue_with_growth.write.save_as_table(
    "GOLD.MONTHLY_REVENUE_BY_CATEGORY", mode="overwrite"
)
print("Monthly revenue analytics created.")

## 8. Verification

Confirm all three analytics tables were populated.

In [ ]:
%%sql -r processed_reviews_check
SELECT COUNT(*) AS TOTAL_REVIEWS,
       ROUND(AVG(SENTIMENT_SCORE), 3) AS AVG_SENTIMENT
FROM GOLD.PROCESSED_REVIEWS

In [ ]:
%%sql -r product_sentiment_check
SELECT PRODUCT_NAME, CATEGORY,
       REVIEW_COUNT, ROUND(AVG_RATING, 2) AS AVG_RATING,
       ROUND(AVG_SENTIMENT, 3) AS AVG_SENTIMENT,
       POSITIVE_REVIEWS, NEGATIVE_REVIEWS
FROM GOLD.PRODUCT_SENTIMENT_SUMMARY
ORDER BY REVIEW_COUNT DESC
LIMIT 10

In [ ]:
%%sql -r revenue_check
SELECT ORDER_MONTH, CATEGORY,
       MONTHLY_REVENUE, ORDER_COUNT, MOM_GROWTH_PCT
FROM GOLD.MONTHLY_REVENUE_BY_CATEGORY
ORDER BY ORDER_MONTH DESC, MONTHLY_REVENUE DESC
LIMIT 10